In [1]:
# Python Standard Library
import sys
from os.path import join

# Community Modules
from tqdm import tqdm
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from IPython.display import display

# My Modules
sys.path.insert(0, "../code")
import measure_signal as ms
import dataset_utils as du
# import spectral_sparklines as sp
from spectral_sparklines import make_dirs, dataset_make_new_noise, dataset_calc_SNR, dataset_sort

In [2]:
new_SNR = 10
# dir_figs = f"../figures/sparklines_compare_SNR{new_SNR}"
rng = np.random.default_rng(1415)
num_sparklines_per_fig = 10

In [3]:
df_dataset = pd.read_parquet("../data/forSNR/deprecated_4/dataset.parquet")
df_signal = pd.read_parquet("../data/forSNR/deprecated_4/signal.parquet")
df_noise = pd.read_parquet("../data/forSNR/deprecated_4/noise.parquet")

wvl, spectra, df_meta = du.unpack_dataset(df_dataset)
num_spectra, num_wvl = spectra.shape
assert num_wvl == wvl.size
print(num_spectra, num_wvl)

3574 1024


In [4]:
subtypes_ID = df_meta["SN Subtype ID"].unique()
subtypes_ID_to_str = df_meta.groupby(by="SN Subtype ID")["SN Subtype"]
subtypes_ID_to_str = subtypes_ID_to_str.apply(lambda x: x.to_numpy()[0])
subtypes_ID_to_str = dict(subtypes_ID_to_str)
# subtypes_ID_to_dir = make_dirs(dir_figs, subtypes_ID_to_str)

In [5]:
new_noise = dataset_make_new_noise(
    new_SNR,
    df_meta,
    num_spectra,
    num_wvl,
    rng)

In [6]:
dataset_calc_SNR(
    new_SNR,
    num_spectra,
    wvl,
    df_dataset,
    spectra,
    new_noise)

Calculating SNR...


100%|██████████| 3574/3574 [03:50<00:00, 15.50it/s]


In [7]:
df_dataset = dataset_sort(df_dataset)

In [17]:
new_noise.shape

(3574, 1024)

In [19]:
closer_look_spectra = []
for subtype_ID, subtype_str in subtypes_ID_to_str.items():
    # dir_subtype_figs = subtypes_ID_to_dir[subtype_ID]
    
    mask = (df_dataset["SN Subtype ID"] == subtype_ID)
    df_subtype = df_dataset[mask].copy(deep=True).reset_index(drop=True)
    
    subtype_num_spectra = mask.sum()
    assert subtype_num_spectra == df_subtype.shape[0]
    print(subtype_ID, subtype_str, subtype_num_spectra)
    for i in tqdm(range(0, subtype_num_spectra, num_sparklines_per_fig)):
        num_plots = np.min((subtype_num_spectra - i, num_sparklines_per_fig))
        
        for j in range(num_plots):
            specsnr = df_subtype.loc[i+j, "specsnr"]

            if (i % 100 == 0) and ((j == 0) or (j == num_plots // 2)):
                df_subtype.loc[i+j, wvl.astype(str)] = specsnr.signal + specsnr.noise
                closer_look_spectra.append(df_subtype.loc[i+j])


        
        # break
    # break

0 Ia-norm 2034


100%|██████████| 204/204 [00:04<00:00, 43.77it/s]


1 Ia-91T 339


100%|██████████| 34/34 [00:00<00:00, 38.44it/s]


2 Ia-91bg 222


...

100%|██████████| 18/18 [00:00<00:00, 40.93it/s]


8 Ic-broad 210


100%|██████████| 21/21 [00:00<00:00, 31.50it/s]


9 IIP 100


100%|██████████| 10/10 [00:00<00:00, 45.58it/s]


In [20]:
df_closer_look = pd.concat(closer_look_spectra, axis=1).T
df_closer_look.drop(columns="specsnr", inplace=True)
df_closer_look.reset_index(inplace=True, drop=True)

In [21]:
df_closer_look

,SN Name,Spectrum Phase,Spectrum Cardinality,SN Subtype,SN Maintype,SN Subtype ID,SN Maintype ID,SNR,S (SNR),N (SNR),...,9885.59,9898.98,9912.39,9925.82,9939.27,9952.73,9966.21,9979.71,9993.24,post_injection_range
0,sn2004dt,-1.0,1,Ia-norm,Ia,0,0,5795.302733,31.618871,0.005456,...,-0.97048,1.066595,0.854407,2.035191,-3.774301,6.944276,-2.374301,-3.680534,-2.390144,21.988639
1,sn2006x,1.3,1,Ia-norm,Ia,0,0,5544.662228,20.202563,0.003644,...,1.21908,-0.291781,0.390276,-0.975217,-0.953956,-2.60962,3.087096,-3.042977,-0.843497,12.648668
2,sn2002he,-3.2,1,Ia-norm,Ia,0,0,2792.582167,13.024933,0.004664,...,0.970256,2.113966,-0.092081,1.51232,0.750895,-0.59536,0.957644,0.418069,1.121849,7.174206
3,sn2004dt,-3.0,1,Ia-norm,Ia,0,0,767.031086,11.448633,0.014926,...,-0.139531,-0.309572,0.196527,-0.726368,-0.536373,-1.337117,-0.407065,0.140905,-0.851154,7.103497
4,sn2003du,20.7,1,Ia-norm,Ia,0,0,1450.331446,8.881133,0.006124,...,-0.356309,-0.589026,-0.784691,-0.069588,-1.27971,-0.99829,0.601248,0.40957,-0.608912,5.940506
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,sn2003dh,30.4,1,Ic-broad,Ic,8,2,31.107475,0.689875,0.022177,...,0.591242,0.450828,0.409041,0.469013,0.487935,0.523724,0.507748,0.611398,0.405972,1.124441
78,sn2013dx,4.6,2,Ic-broad,Ic,8,2,3.599507,0.331879,0.092201,...,0.589459,0.58805,0.506044,0.532571,0.58653,0.561613,0.591733,0.535322,0.59087,0.517859
79,sn2013dx,6.4,1,Ic-broad,Ic,8,2,4.73554,0.215674,0.045544,...,0.618346,0.598046,0.615582,0.597402,0.594191,0.612268,0.592902,0.627685,0.597861,0.46275
80,sn2004et,47.4,1,IIP,II,9,3,3698.114474,22.91162,0.006195,...,1.071375,-0.711781,2.236526,-0.200217,-1.603493,0.505256,-1.802164,2.681163,1.089652,13.512432


In [22]:
df_closer_look.to_parquet("../data/forSNR/closer_look.parquet")

In [24]:
df_closer_look.loc[0, "minima_i"]

nan